In [10]:
# 1-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_1_1.csv')
# print(df.info())
# print(df.head())

df = df[(df['ProductA가격'] != 0) & (df['ProductB가격'] != 0)]
df['합계'] = np.abs(df['ProductA가격'] - df['ProductB가격'])
print(np.round(df.groupby('도시명')['합계'].mean().max(), 1))

16333.3


In [9]:
# 1-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_1_2.csv')
# print(df.info())
# print(df.head())

df['BMI'] = df['Weight_kg'] / (df['Height_cm'] / 100)**2
print((df[df['BMI']<18.5].count()) + (df[df['BMI']>=25].count()))

Height_cm    74
Weight_kg    74
BMI          74
dtype: int64


In [46]:
# 1-3
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_1_3.csv')
# print(df.info())
# print(df.head())
df['생산된 제품수'] = df['products_made_domestic'] + df['products_made_international']
df['판매된 제품수'] = df['products_sold_domestic'] + df['products_sold_international']
df['순 생산량'] = df['생산된 제품수'] - df['판매된 제품수']
# df.groupby('year')[['순 생산량', 'factory']].max()
# df[df['factory'] == 'Factory E']['순 생산량'].sum()

df1 = df.groupby(['year', 'factory'])['순 생산량'].sum().reset_index()
df2 = df1.groupby('year')['순 생산량'].idxmax()
df3 = df1.loc[df2]
print(df3['순 생산량'].sum())

9130


In [104]:
# 2
import numpy as np
import pandas as pd
train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_2_test.csv')
# print(train_df.info())
# print(train_df.head())
# print(test_df.info())

train_y = train_df['DBP']
train_X = train_df.drop(columns=['level_0', 'ID', 'DBP'])
test_y = test_df['DBP']
test_X = test_df.drop(columns=['level_0', 'ID', 'DBP'])

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import root_mean_squared_error, get_scorer_names
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
# neg_root_mean_squared_error
# SimpleImputer(strategy='most_frequent')

train_X, valid_X, train_y, valid_y = train_test_split(train_X, train_y, test_size=0.3)
# print(train_X.shape, train_y.shape, valid_X.shape, valid_y.shape)

train_X_obj = train_X.select_dtypes('object').columns.tolist()
train_X_num = train_X.select_dtypes('number').columns.tolist()
valid_X_obj = valid_X.select_dtypes('object').columns.tolist()
valid_X_num = valid_X.select_dtypes('number').columns.tolist()
test_X_obj = test_X.select_dtypes('object').columns.tolist()
test_X_num = test_X.select_dtypes('number').columns.tolist()

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
imputer = SimpleImputer(strategy='most_frequent')

train_X_impute = imputer.fit_transform(train_X[train_X_obj])
valid_X_impute = imputer.transform(valid_X[valid_X_obj])
test_X_impute = imputer.transform(test_X[test_X_obj])

train_X_one = encoder.fit_transform(train_X_impute)
valid_X_one = encoder.transform(valid_X_impute)
test_X_one = encoder.transform(test_X_impute)

train_X = np.concatenate([train_X_one, train_X[train_X_num]], axis=1)
valid_X = np.concatenate([valid_X_one, valid_X[valid_X_num]], axis=1)
test_X = np.concatenate([test_X_one, test_X[test_X_num]], axis=1)

rf=RandomForestRegressor()
rf.fit(train_X, train_y)
pred1 = rf.predict(valid_X)
print(root_mean_squared_error(valid_y, pred1))

params={
    'max_depth': [10,20,30],
    'min_samples_split': [2,5,10]
}
cv = GridSearchCV(estimator=rf, param_grid=params, scoring='neg_root_mean_squared_error', cv=3)
cv.fit(train_X, train_y)
pred2 = cv.predict(valid_X)
print(root_mean_squared_error(valid_y, pred2))

pred3 = cv.predict(test_X)
pd.DataFrame(pred3, columns=['pred']).to_csv('result.csv', index=False)

12.47944758745002
12.432667543021983


In [57]:
# 3-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_3_1.csv')
# print(df.info())
# print(df.head())
print(df['work_hours'].mean().round(3))
print(np.round(df['work_hours'].std(),3))
mu=df['work_hours'].mean().round(3)
sigma=np.round(df['work_hours'].std(),3)
from scipy.stats import kstest
print(kstest(df['work_hours'], "norm", args=(mu, sigma)).statistic.round(3))
print(kstest(df['work_hours'], "norm", args=(mu, sigma)).pvalue.round(4) < 0.05)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   work_hours  100 non-null    float64
dtypes: float64(1)
memory usage: 932.0 bytes
None
   work_hours
0   10.646079
1    8.600236
2    9.468107
3   11.361340
4   10.801337
8.09
1.519
0.064
False


In [73]:
# 3-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_3_2.csv')
# print(df.info())
# print(df.head())

import statsmodels.api as sm
import statsmodels.formula.api as smf
model = smf.ols("price ~ area + rooms + age", data=df).fit()
print(model.summary())
print(model.params[1:].idxmax())
model.pvalues[1:] < 0.05
# (model.pvalues[1:] < 0.05).sum()

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.708
Model:                            OLS   Adj. R-squared:                  0.699
Method:                 Least Squares   F-statistic:                     77.65
Date:                Mon, 18 May 2026   Prob (F-statistic):           1.41e-25
Time:                        13:29:33   Log-Likelihood:                -1055.4
No. Observations:                 100   AIC:                             2119.
Df Residuals:                      96   BIC:                             2129.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   5.395e+04   6081.703      8.871      0.0

area     True
rooms    True
age      True
dtype: bool